### Copyright (C) Infineon Technologies AG 2025
 
Copyright (c) 2025, Infineon Technologies AG, or an affiliate of Infineon Technologies AG. All rights reserved.
This software, associated documentation and materials ("Software") is owned by Infineon Technologies AG or one of its affiliates ("Infineon") and is protected by and subject to worldwide patent protection, worldwide copyright laws, and international treaty provisions. Therefore, you may use this Software only as provided in the license agreement accompanying the software package from which you obtained this Software. If no license agreement applies, then any use, reproduction, modification, translation, or compilation of this Software is prohibited without the express written permission of Infineon.

Disclaimer: UNLESS OTHERWISE EXPRESSLY AGREED WITH INFINEON, THIS SOFTWARE IS PROVIDED AS-IS, WITH NO WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING, BUT NOT LIMITED TO, ALL WARRANTIES OF NON-INFRINGEMENT OF THIRD-PARTY RIGHTS AND IMPLIED WARRANTIES SUCH AS WARRANTIES OF FITNESS FOR A SPECIFIC USE/PURPOSE OR MERCHANTABILITY. Infineon reserves the right to make changes to the Software without notice. You are responsible for properly designing, programming, and testing the functionality and safety of your intended application of the Software, as well as complying with any legal requirements related to its use. Infineon does not guarantee that the Software will be free from intrusion, data theft or loss, or other breaches ("Security Breaches"), and Infineon shall have no liability arising out of any Security Breaches. Unless otherwise explicitly approved by Infineon, the Software may not be used in any application where a failure of the Product or any consequences of the use thereof can reasonably be expected to result in personal injury.

# Conversion Debug Notebook

Interactively run the `nn2ifx` pipeline (ONNX → C → compile → emulate) for a
single hardware target, or compare a model across all supported platforms:

| Target | Toolchain | Emulator |
| --- | --- | --- |
| `tc3xx` | AURIX™ TC3x GCC | QEMU (+ optional TSIM) |
| `tc4dx` | AURIX™ TC4x GCC | QEMU (+ optional TSIM) |
| `tc4dx_ppu` | AURIX™ TC4x PPU (ARC clang) | nSIM |
| `arm_m4` | Arm® Cortex®-M4 GCC | QEMU |

## Notebook Structure

1. **Setup and Toolchain Inspection**
2. **Input Model and Test-Data Configuration**
3. **Single-Target Conversion**
4. **Generated-Artifact Inspection**
5. **Cross-Platform Conversion and Benchmark Comparison**
6. **Troubleshooting and Output Interpretation**

# Setup

In [ ]:
import os
import sys
from pathlib import Path

# The conversion service application lives here in the container image.
sys.path.insert(0, "/home/ubuntu/app")

from model_converter import ModelConverter, TARGETS

print("🚀 Neural Network Conversion Tools - Debug Mode")
print("=" * 50)
print("Workspace: /home/ubuntu/workspace")
print(f"Supported targets: {TARGETS}")

# Tool paths are configured via environment variables consumed by nn2ifx.config
for var in (
    "ONNX2C_CMD",
    "TRICORE_GCC_CMD",
    "ARM_GCC_CMD",
    "ARC_CLANG_CMD",
    "NSIM_CMD",
):
    print(f"  - {var}: {os.environ.get(var, '(unset)')}")

## Input Model and Test-Data Configuration

This utility notebook does not train a model or use a fixed dataset. It consumes user-provided conversion inputs:

- `model.onnx`: the ONNX model to convert.
- `test_data_set/input_0.pb`: serialized representative input data.
- `test_data_set/output_0.pb`: expected reference output data used to validate conversion.
- `TARGET`: one of `tc3xx`, `tc4dx`, `tc4dx_ppu`, or `arm_m4`.
- `OUTPUT_DIR`: destination for generated C code, binaries, logs, timing data, and result files.

Place `model.onnx` and the `test_data_set/` directory below the configured workspace path. The notebook discovers the first matching model and test-data directory.

In [ ]:
MODEL_PATH = "/home/ubuntu/workspace"  # Where your model.onnx is located
TARGET = "tc4dx"  # tc3xx, tc4dx, tc4dx_ppu, arm_m4
OUTPUT_DIR = "/home/ubuntu/out"
PROFILE = False  # Per-node instruction profiling (TriCore/Arm)
PROFILER = "tsim"  # TriCore timing backend: "tsim" (default) or "qemu"

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

model_files = list(Path(MODEL_PATH).glob("**/model.onnx"))
test_data_dirs = list(Path(MODEL_PATH).glob("**/test_data_set"))

if not model_files:
    print("❌ No model.onnx found! Upload your model to", MODEL_PATH)
else:
    MODEL = model_files[0]
    TEST_DATA = test_data_dirs[0] if test_data_dirs else MODEL.parent / "test_data_set"
    print(f"✅ Model:     {MODEL}")
    print(f"✅ Test data: {TEST_DATA}")
    print(f"   Target:   {TARGET}")

## Single-Target Conversion and Generated Artifacts

The selected target runs the `nn2ifx` pipeline from ONNX to generated C, compilation, and emulation. Outputs are written to `OUTPUT_DIR/<target>/` and include `model.c`, `main.c`, `model.elf`, `model.md`, `pipeline.log`, and `results.json`. The conversion result is printed immediately so failures and target-specific metrics are visible.

In [ ]:
converter = ModelConverter(
    model=MODEL,
    test_data=TEST_DATA,
    out_dir=OUTPUT_DIR,
    profile=PROFILE,
    profiler=PROFILER,
)

result = converter.run(TARGET)
print("\n📊 results.json:")
for key, value in result.to_dict().items():
    print(f"  {key}: {value}")

In [ ]:
# Inspect the generated C code (model.c)
from model_converter import resolve_target

model_c = Path(OUTPUT_DIR) / resolve_target(TARGET) / "model.c"
if model_c.exists():
    print(f"✅ Generated C code: {model_c} ({model_c.stat().st_size} bytes)\n")
    print("".join(model_c.read_text().splitlines(keepends=True)[:30]))
else:
    print("❌ model.c not found — check pipeline.log")

## Generated-Artifact Inspection

Inspect `model.c` to confirm that code generation completed. For failures, start with `pipeline.log` and `results.json`; check the selected target, toolchain environment variables, model path, and serialized reference data before rerunning the conversion.

## Cross-Platform Conversion and Benchmark Comparison

Run the same model and serialized test data on every supported target. The notebook writes `comparison.txt` and `comparison.json` to `OUTPUT_DIR` and displays the resulting conversion and execution comparison.

In [ ]:
converter.compare()  # defaults to all targets; writes comparison.txt/json
print((Path(OUTPUT_DIR) / "comparison.txt").read_text())

## Troubleshooting and Output Interpretation

- If no model is found, verify that `model.onnx` is below `MODEL_PATH`.
- If validation fails, verify `input_0.pb` and `output_0.pb` and confirm that they were generated with the same model and preprocessing.
- If compilation fails, inspect the reported toolchain environment variable and `pipeline.log` for the selected target.
- Use `results.json` for single-target timing and status details, and `comparison.txt` / `comparison.json` for cross-platform results.

This notebook produces no trained model or fixed dataset. Its reproducible inputs are the user-provided `model.onnx`, `test_data_set/input_0.pb`, and `test_data_set/output_0.pb` files.

## References and Generated Artifacts

- **Input source:** User-provided ONNX model and serialized input/output files; this utility notebook does not train a model or use an external dataset.
- **Generated artifacts:** Converted C source, binaries, logs, `results.json`, validation results, `comparison.txt`, and `comparison.json` are saved under `out/<target>/` and `out/`.
- **Deployment targets:** AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4.